# VGG Transfer Learning with Metadata (Multimodal Model)


In [5]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [6]:
!pip install tensorflow

In [7]:
import pandas as pd
import numpy as np
import os, zipfile
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.applications import VGG16
from tensorflow.keras.preprocessing.image import load_img, img_to_array
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, Dropout, GlobalAveragePooling2D, Input, Concatenate
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.utils import to_categorical


In [8]:
# Load CSV
train_df = pd.read_csv('/content/drive/MyDrive/train_data.csv')
train_df.columns = train_df.columns.str.strip()

# Extract photo_id (if needed)
train_df['photo_id'] = train_df['preprocessed_path'].apply(lambda x: os.path.basename(x))

# Extract ZIP of preprocessed images
zip_path = '/content/drive/MyDrive/preprocessed_photos.zip'
extract_path = '/content/dataset'
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

In [9]:
import pandas as pd

# Load CSV
train_df = pd.read_csv('/content/drive/MyDrive/train_data.csv')

# Show columns
print("Columns:", train_df.columns.tolist())

# Show first few rows
print(train_df.head())

Columns: ['photo_id', 'label', 'label_encoded', 'preprocessed_path']
                 photo_id   label  label_encoded  \
0  38uF8Xq04Gitx84x_o24Tw  inside              2   
1  VXpYAvJJFxA39v3ytyAHJg    food              1   
2  IkXGsVerw1GXPenUe5WEkA  inside              2   
3  VGrj-6peaeoytB8jSwOq1g    food              1   
4  WbdvpVcuXuBz4FYjmf8zZA  inside              2   

                                preprocessed_path  
0  preprocessed_photos/38uF8Xq04Gitx84x_o24Tw.jpg  
1  preprocessed_photos/VXpYAvJJFxA39v3ytyAHJg.jpg  
2  preprocessed_photos/IkXGsVerw1GXPenUe5WEkA.jpg  
3  preprocessed_photos/VGrj-6peaeoytB8jSwOq1g.jpg  
4  preprocessed_photos/WbdvpVcuXuBz4FYjmf8zZA.jpg  


In [10]:
# Test access to the preprocessed_path column
train_df['preprocessed_path'].head()

,preprocessed_path
0,preprocessed_photos/38uF8Xq04Gitx84x_o24Tw.jpg
1,preprocessed_photos/VXpYAvJJFxA39v3ytyAHJg.jpg
2,preprocessed_photos/IkXGsVerw1GXPenUe5WEkA.jpg
3,preprocessed_photos/VGrj-6peaeoytB8jSwOq1g.jpg
4,preprocessed_photos/WbdvpVcuXuBz4FYjmf8zZA.jpg


In [11]:
train_df.columns = train_df.columns.str.strip()

In [12]:
train_df['preprocessed_path'].head()

,preprocessed_path
0,preprocessed_photos/38uF8Xq04Gitx84x_o24Tw.jpg
1,preprocessed_photos/VXpYAvJJFxA39v3ytyAHJg.jpg
2,preprocessed_photos/IkXGsVerw1GXPenUe5WEkA.jpg
3,preprocessed_photos/VGrj-6peaeoytB8jSwOq1g.jpg
4,preprocessed_photos/WbdvpVcuXuBz4FYjmf8zZA.jpg


In [13]:
train_df = train_df.sample(n=25000, random_state=42).reset_index(drop=True)

# Encode labels
train_df['label_id'] = train_df['label'].astype('category').cat.codes
num_classes = train_df['label_id'].nunique()

# Full image paths
train_df['full_path'] = train_df['preprocessed_path'].apply(lambda x: os.path.join(extract_path, x))

In [14]:
from sklearn.model_selection import train_test_split
from tensorflow.keras.utils import to_categorical

# Split
train_paths, val_paths, train_labels, val_labels = train_test_split(
    train_df['full_path'], train_df['label_id'],
    test_size=0.2, stratify=train_df['label_id'], random_state=42
)

# One-hot encode labels
train_labels_cat = to_categorical(train_labels, num_classes=num_classes)
val_labels_cat = to_categorical(val_labels, num_classes=num_classes)

In [15]:
import tensorflow as tf
from tensorflow.keras.applications.vgg16 import preprocess_input

def process_image(path, label):
    image = tf.io.read_file(path)
    image = tf.image.decode_jpeg(image, channels=3)
    image = tf.image.resize(image, [224, 224])
    image = preprocess_input(image)
    return image, label

def create_dataset(paths, labels, batch_size=32, shuffle=True):
    dataset = tf.data.Dataset.from_tensor_slices((paths, labels))
    if shuffle:
        dataset = dataset.shuffle(buffer_size=1000)
    dataset = dataset.map(process_image, num_parallel_calls=tf.data.AUTOTUNE)
    dataset = dataset.batch(batch_size).prefetch(tf.data.AUTOTUNE)
    return dataset

train_ds = create_dataset(train_paths.values, train_labels_cat, batch_size=32)
val_ds = create_dataset(val_paths.values, val_labels_cat, batch_size=32, shuffle=False)

In [16]:
from tensorflow.keras.applications import VGG16
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, GlobalAveragePooling2D, Dense, Dropout
from tensorflow.keras.optimizers import Adam

# VGG16 model
img_input = Input(shape=(224, 224, 3))
base_model = VGG16(include_top=False, weights='imagenet', input_tensor=img_input)

for layer in base_model.layers:
    layer.trainable = False

x = GlobalAveragePooling2D()(base_model.output)
x = Dropout(0.3)(x)
x = Dense(128, activation='relu')(x)
x = Dropout(0.3)(x)
output = Dense(num_classes, activation='softmax')(x)

model = Model(inputs=img_input, outputs=output)
model.compile(optimizer=Adam(learning_rate=1e-4), loss='categorical_crossentropy', metrics=['accuracy'])

model.summary()

58889256/58889256 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_conv1 (Conv2D)           │ (None, 224, 224, 64)   │         1,792 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_conv2 (Conv2D)           │ (None, 224, 224, 64)   │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_pool (MaxPooling2D)      │ (None, 112, 112, 64)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_conv1 (Conv2D)           │ (None, 112, 112, 128)  │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_conv2 (Conv2D)           │ (None, 112, 112, 128)  │       147,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_pool (MaxPooling2D)      │ (None, 56, 56, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv1 (Conv2D)           │ (None, 56, 56, 256)    │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv2 (Conv2D)           │ (None, 56, 56, 256)    │       590,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv3 (Conv2D)           │ (None, 56, 56, 256)    │       590,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_pool (MaxPooling2D)      │ (None, 28, 28, 256)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv1 (Conv2D)           │ (None, 28, 28, 512)    │     1,180,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv2 (Conv2D)           │ (None, 28, 28, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv3 (Conv2D)           │ (None, 28, 28, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_pool (MaxPooling2D)      │ (None, 14, 14, 512)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv1 (Conv2D)           │ (None, 14, 14, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv2 (Conv2D)           │ (None, 14, 14, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv3 (Conv2D)           │ (None, 14, 14, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_pool (MaxPooling2D)      │ (None, 7, 7, 512)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 512)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │        65,664 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 5)              │           645 │
└─────────────────────────────────┴────────────────────────┴─────────────

 Total params: 14,780,997 (56.39 MB)

 Trainable params: 66,309 (259.02 KB)

 Non-trainable params: 14,714,688 (56.13 MB)

In [17]:
from tensorflow.keras.callbacks import EarlyStopping

model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10,
    callbacks=[EarlyStopping(patience=2, restore_best_weights=True)]
)

Epoch 1/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 146s 211ms/step - accuracy: 0.6700 - loss: 2.5659 - val_accuracy: 0.9102 - val_loss: 0.3603
Epoch 2/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 203s 233ms/step - accuracy: 0.8509 - loss: 0.7413 - val_accuracy: 0.9270 - val_loss: 0.2445
Epoch 3/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 200s 230ms/step - accuracy: 0.8755 - loss: 0.4970 - val_accuracy: 0.9288 - val_loss: 0.2115
Epoch 4/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 202s 230ms/step - accuracy: 0.8819 - loss: 0.4064 - val_accuracy: 0.9324 - val_loss: 0.2018
Epoch 5/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 203s 232ms/step - accuracy: 0.8963 - loss: 0.3437 - val_accuracy: 0.9360 - val_loss: 0.1969
Epoch 6/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 144s 231ms/step - accuracy: 0.8975 - loss: 0.3262 - val_accuracy: 0.9362 - val_loss: 0.1898
Epoch 7/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 202s 231ms/step - accuracy: 0.9073 - loss: 0.2874 - val_accuracy: 0.9392 - val_loss: 0.1873
Epoch 8/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 187s 206ms/step - accuracy: 0.9128 -

In [19]:
# Unfreeze the last 4 layers of the VGG16 base model
for layer in base_model.layers[-4:]:
    layer.trainable = True

# Re-compile with a lower learning rate for fine-tuning
model.compile(optimizer=Adam(learning_rate=1e-5), loss='categorical_crossentropy', metrics=['accuracy'])

# Fine-tune the model
model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=5,
    callbacks=[EarlyStopping(patience=2, restore_best_weights=True)]
)


Epoch 1/5
625/625 ━━━━━━━━━━━━━━━━━━━━ 175s 270ms/step - accuracy: 0.9432 - loss: 0.1684 - val_accuracy: 0.9544 - val_loss: 0.1374
Epoch 2/5
625/625 ━━━━━━━━━━━━━━━━━━━━ 199s 270ms/step - accuracy: 0.9524 - loss: 0.1343 - val_accuracy: 0.9590 - val_loss: 0.1299
Epoch 3/5
625/625 ━━━━━━━━━━━━━━━━━━━━ 186s 245ms/step - accuracy: 0.9618 - loss: 0.1060 - val_accuracy: 0.9592 - val_loss: 0.1277
Epoch 4/5
625/625 ━━━━━━━━━━━━━━━━━━━━ 168s 268ms/step - accuracy: 0.9684 - loss: 0.0850 - val_accuracy: 0.9584 - val_loss: 0.1328
Epoch 5/5
625/625 ━━━━━━━━━━━━━━━━━━━━ 204s 271ms/step - accuracy: 0.9763 - loss: 0.0687 - val_accuracy: 0.9606 - val_loss: 0.1328


In [20]:
# Evaluate on validation set
loss, acc = model.evaluate(val_ds)
print(f"Validation Accuracy: {acc:.4f}")

157/157 ━━━━━━━━━━━━━━━━━━━━ 27s 173ms/step - accuracy: 0.9577 - loss: 0.1320
Validation Accuracy: 0.9592
